In [1]:
import os, time, glob, shutil, random, requests
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta, timezone
import os, time, random
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import Chrome, ChromeOptions
import pandas as pd
from itertools import cycle
import io
from io import BytesIO

Сохранение переводов

In [ ]:
def to_dt_ms(dt: datetime) -> int:
    return int(dt.timestamp() )

amount=1
save_folder = r"C:\Users\User\my\py\practise\solana\downloads1"



def save_transfer(address,from_time,to_time):
    url = (
        "https://api-v2.solscan.io/v2/account/transfer/export"
        f"?address={address}"
        "&remove_spam=true"
        "&exclude_amount_zero=true"
        "&token=So11111111111111111111111111111111111111111"
        f"&from_time={from_time}"
        f"&to_time={to_time}"
        f"&amount[]={amount}"
    )
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
        "Origin": "https://solscan.io",
        "Referer": "https://solscan.io/",
    }
    response = requests.get(url, headers=headers)
    df = pd.read_csv(BytesIO(response.content))
    content_disposition = response.headers.get("Content-Disposition", "")
    base_name = "export.csv"
    if "filename=" in content_disposition:
        base_name = content_disposition.split("filename=")[-1].strip('"')


    base_name_no_ext, _ = os.path.splitext(base_name)
    base_name = (
        f"{base_name_no_ext}"
        f"_addr-{address}"
        f"_from-{from_time}"
        f"_to-{to_time}"
        f"_amt-{amount}.csv"
    )


    address_folder = os.path.join(save_folder, address)
    os.makedirs(address_folder, exist_ok=True)
    filename = os.path.join(address_folder, base_name)


    with open(filename, "wb") as f:
        f.write(response.content)

    dt = datetime.fromtimestamp(to_time , tz=timezone.utc)
    print(f"Сохранили 1000 по {address} с {dt} в {filename}")
    return df

def save_transfer_proxi(address,from_time,to_time,proxy):
    url = (
        "https://api-v2.solscan.io/v2/account/transfer/export"
        f"?address={address}"
        "&remove_spam=true"
        "&exclude_amount_zero=true"
        "&token=So11111111111111111111111111111111111111111"
        f"&from_time={from_time}"
        f"&to_time={to_time}"
        f"&amount[]={amount}"
    )
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
        "Origin": "https://solscan.io",
        "Referer": "https://solscan.io/",
    }
    proxies_dict = {
    "http":  f"http://{proxy}",
    "https": f"http://{proxy}",
}
    response = requests.get(url, headers=headers, proxies=proxies_dict,timeout=4)

    df = pd.read_csv(BytesIO(response.content))
    content_disposition = response.headers.get("Content-Disposition", "")
    base_name = "export.csv"
    if "filename=" in content_disposition:
        base_name = content_disposition.split("filename=")[-1].strip('"')


    base_name_no_ext, _ = os.path.splitext(base_name)
    base_name = (
        f"{base_name_no_ext}"
        f"_addr-{address}"
        f"_from-{from_time}"
        f"_to-{to_time}"
        f"_amt-{amount}.csv"
    )


    address_folder = os.path.join(save_folder, address)
    os.makedirs(address_folder, exist_ok=True)
    filename = os.path.join(address_folder, base_name)

    # Сохраняем файл
    with open(filename, "wb") as f:
        f.write(response.content)

    dt = datetime.fromtimestamp(to_time , tz=timezone.utc)
    print(f"Сохранили 1000 ; использовали {proxy} по {address} с {dt} в {filename}")
    return df

Тут работа с прокси, которые были лично у меня, код ниже не нужен. Задача просто иметь такой список (proxy_list).

In [ ]:
proxies=pd.read_excel(r"C:\Users\User\my\py\practise\solana\proxylist5.xlsx")


In [4]:
url = "https://api.best-proxies.ru/proxylist.csv?key=b11e135a1f44c9a7e977e92bf17fa9dd&limit=0"
try:
    r = requests.get(url, timeout=5)
    r.raise_for_status()

    proxies = pd.read_csv(
        io.StringIO(r.text),
        sep=";",          # важно: разделитель ;
        quotechar='"',    # кавычки как в примере
        engine="python"
    )
    proxies.to_csv(r"C:\Users\User\my\py\practise\solana\proxylist_renewable.xlsx")
except:
    print('Используем старые proxi')
    proxies=pd.read_csv(r"C:\Users\User\my\py\practise\solana\proxylist_renewable.xlsx")
proxies["port"] = pd.to_numeric(proxies["port"], errors="coerce")
proxies = proxies.dropna(subset=["port"])
proxies["port"] = proxies["port"].astype(int)
proxies["full"] = proxies["ip"] + ":" + proxies["port"].astype(str)
proxies.sort_values(by=['response, msec'])
proxies=proxies[proxies['country code']!='RU']
columns = ["full", 'response, msec', 'country code']

df_small = proxies[columns]

df_small=df_small.sort_values(by="response, msec")
df_small=df_small[100:]
proxy_list = proxies["full"].tolist()
proxy_list=proxy_list[100:]

In [8]:
proxies

,ip,port,country code,city,region,country,level,http,https,socks4,...,instagram,telegram,avito,mailru,real_ip,"response, msec",good checks,bad checks,hostname,full
0,87.95.146.131,80,FI,Kuopio,North Savo,Финляндия,1,1,0,0,...,0,0,0,0,87.95.146.131,151,23,38,87-95-146-131.bb.dnainternet.fi,87.95.146.131:80
2,79.110.198.37,8080,PL,Wroclaw,Lower Silesia,Польша,1,1,0,0,...,0,0,0,0,79.110.198.37,151,3222,4774,caril.static.korbank.pl,79.110.198.37:8080
3,65.20.237.105,8080,IQ,Karbala,Muhafazat Karbala',Ирак,1,1,0,0,...,0,0,0,0,65.20.237.105,151,2541,3384,65.20.237.105,65.20.237.105:8080
5,62.60.131.253,31753,IN,Navi Mumbai,Maharashtra,Индия,1,0,0,0,...,0,0,0,0,195.250.31.211,137,4,2,748359-host.webnoxdigital.com,62.60.131.253:31753
6,62.60.131.205,18471,TW,Taipei,Taipei City,Тайвань,1,0,0,0,...,0,0,0,0,125.228.47.114,137,3,0,125.228.47.114,62.60.131.205:18471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3136,152.53.54.151,29245,US,Arlington,Massachusetts,США,1,0,0,0,...,0,0,0,0,73.227.183.93,331,3,0,c-73-227-183-93.hsd1.ma.comcast.net,152.53.54.151:29245
3137,152.53.54.151,22391,AU,Perth,Western Australia,Австралия,1,0,0,0,...,0,0,0,0,115.131.175.57,323,3,1,115.131.175.57,152.53.54.151:22391
3138,152.53.54.151,21955,US,Cleveland Heights,Ohio,США,1,0,0,0,...,0,1,0,0,76.35.212.154,107,3,2,syn-076-035-212-154.res.spectrum.com,152.53.54.151:21955
3139,152.53.54.151,20918,FR,Caen,Calvados,Франция,1,0,0,0,...,1,1,1,0,91.160.102.68,208,3,1,91.160.102.68,152.53.54.151:20918


In [36]:
proxies=pd.read_csv(r"C:\Users\User\my\py\practise\solana\proxylist_renewable.xlsx")
proxies['country code'].value_counts()

country code
US    833
FR    304
GB    240
ES    239
DE    236
     ... 
KY      1
SV      1
TN      1
LU      1
GG      1
Name: count, Length: 91, dtype: int64

In [8]:
proxy_list

['36.255.98.161:4308',
 '49.49.46.4:8080',
 '62.60.131.181:4509',
 '85.239.157.25:80',
 '89.43.132.247:8080',
 '102.218.195.5:8080',
 '103.166.159.147:8080',
 '103.171.183.222:8097',
 '103.195.65.140:8080',
 '103.217.219.51:1111',
 '105.214.91.155:5678',
 '110.235.246.228:1080',
 '113.192.12.73:8085',
 '115.74.155.152:1080',
 '117.1.59.145:1080',
 '120.255.40.222:22222',
 '124.248.190.48:1080',
 '157.20.207.115:8080',
 '160.19.135.12:5678',
 '163.61.187.142:8080',
 '171.4.1.165:8080',
 '171.234.166.171:1080',
 '182.253.248.164:8080',
 '202.62.42.92:1080',
 '202.191.127.34:1121',
 '203.189.150.44:1080',
 '223.205.100.102:4145',
 '152.53.171.242:16276',
 '152.53.171.242:16294',
 '152.53.171.242:16303',
 '46.28.88.19:28818',
 '152.53.171.242:16330',
 '152.53.171.242:16347',
 '152.53.171.242:16350',
 '152.53.171.242:16363',
 '152.53.171.242:16378',
 '152.53.171.242:16379',
 '152.53.171.242:16381',
 '152.53.171.242:16383',
 '152.53.171.242:16387',
 '152.53.171.242:16388',
 '152.53.171.242:1

In [ ]:
filtered_adresses={'22Wnk8PwyWZV7BfkZGJEKT9jGGdtvu7xY6EXeRh7zkBa', 
                   '2AQdpHJ2JpcEgPiATUXjQxA8QmafFegfQwSLWSprPicm',
                    '2kmT6TAgwy2aPt4wtiSmmosyTZ8kVRSXwXxtbgJ8QfLM', 
                    '2nE99H3e9gfbxKaGodnTwixMgZwgTtKg45aoCFwdEfJS',
                    '2qo8jvuc49pFmTjmUHLiARSV6ppPTaE7gw27ZJ6DnNZy', 
                    '2YERiDP8JqbmKDDUhPQtEfWozafQm1cdwi6KrmYo2app',
                    '2YxQCXt9spMwoQZiwFLwdjHtscVvXi4nmxCkQCD6Rvgg',
                           '32cT9eAwkEvAk631rUcUAbXVFPg21DaAXzGiz9AqHTVE',
                             '3jn81xbBK95zf5A1K4jRMzTicWxKM6X8BzYWzhPtFjuF',
                               '3RgFXirEPVEqpuoCpz4wbwc2hv2efTvQHLVddthG5n79',
                                 '4NyK1AdJBNbgaJ9EsKz3J4rfeHsuYdjkTPg3JaNdLeFw',
                                   '555oNTKdRECgyLn8fBvySoN6hXMCszFq1Y4oea9p3ZFB',
                                     '57vSaRTqN9iXaemgh4AoDsZ63mcaoshfMK8NP3Z5QNbs',
                                       '59L2oxymiQQ9Hvhh92nt8Y7nDYjsauFkdb3SybdnsG6h',
                                         '5g7yNHyGLJ7fiQ9SN9mf47opDnMjc585kqXWt6d7aBWs',
                                           '5LZkATrLwHYCQj2YuVbjjgsDZzBk6YfL4pFQRJmtboT2',
                                             '5PAhQiYdLBd6SVdjzBQDxUAEFyDdF5ExNPQfcscnPRj5',
                                               '5VCwKtCXgCJ6kit5FybXjvriW3xELsFDhYrPSqtJNmcD',
                                                 '6FEVkH17P9y8Q9aCkDdPcMDjvj7SVxrTETaYEm8f51Jy',
                                                   '6LY1JzAFVZsP2a2xKrtU6znQMQ5h4i7tocWdgrkZzkzF',
                                                     '6mearRPb9Bma9Ad9wCKnGtVbTeqDyZQ6PPxDRpBUNdJU',
                                                       '6Nsr6vKFhiqu126FThNjAKpzvz2xMAUHHmgYLDATDkab',
                                                         '6VGNcrNuhwCwWZDQoA9QoyJxKtb4FEGLnieScUnLbeKJ',
                                                           '6yhYE7xUjy9hPeYDFqyVBYjnKHPXZZsg9LG48t3vMSSU',
                                                             '7cAui6ADtxLnpRr2wYvwJWTkzwgmVF2LYKnjKTLx4xR8',
                                                               '7EUv2zT1gufS4R6Xz9MsSdM9zzyuPMcpoe9UwQxZgjUh',
                                                                 '7mhcgF1DVsj5iv4CxZDgp51H6MBBwqamsH1KnqXhSRc5',
                                                                   '7ReR6syi6gr7qUrKCL1FB9VFzGhVgHwLJ8wtfNtH9Mv4',
                                                                     '836euAuAeEWdjHFZ87vKhR6n4Vu1V22yMvj1MyzCdVU3',
                                                                       '88xTWZMeKfiTgbfEmPLdsUCQcZinwUfk25EBQZ21XMAZ',
                                                                         '8jjpnSUhm5UYghznhcdkwwEUFyke6z5TchHEtUUgW8hX',
                                                                           '8JUEoHNMbWM4eegcMvx6ZQJt3V3fbo8JNjpWiCtJgxfF',
                                                                             '8Mm46CsqxiyAputDUp2cXHg41HE3BfynTeMBDwzrMZQH',
                                                                               '8vUsFSgyoDgXWdBBjST6Kny3feT2UPEMuEzY2WCRgLtE',
                                                                                 '8wM44Ryv9DFCSfkgUnPEPgnsc53arT4cnmXL6LnnC4UW',
                                                                                   '9obNtb5GyUegcs3a1CbBkLuc5hEWynWfJC6gjz5uWQkE',
                                                                                     '9un5wqE3q4oCjyrDkwsdD48KteCJitQX5978Vh7KKxHo',
                                                                                       '9VWD3xbXWtC2aGN1mQcVFwZMRPSjJfknNYq4DqGQ7B8B',
                                                                                         '9Z7S8vCj6nDbK9t4m4AU3vZpKm4UufHAwpmRYyKgZf7r',
                                                                                           '9ZifroknFoYu4r6DUk6nYoJiUQnEyyoUyeAwjXbPoL2x',
                                                                                             'A5QtrjyeJLZ62HqSZAmiqAboiBJZX3eQbCe4QSLBGXXN',
                                                                                               'A77HErqtfN1hLLpvZ9pCtu66FEtM8BveoaKbbMoZ4RiR',
                                                                                                 'AafGzY9eiC5Ud3YFZQwkaKApp48cVBAT2kksGvEjhUvH',
                                                                                                   'AaFm2LPX8NUKXe64JaxcRNUc8QPGYCxrPG1HjHcTTGAK',
                                                                                                     'AC5RDfQFmDS1deWZos921JfqscXdByf8BKHs5ACWjtW2',
                                                                                                       'ADqZYsW9SVZ6bCSGDqfBiPkfaZuQKGg79MA357otX8Ud',
                                                                                                         'AEZoku1fLfUz5JYJ3kJ5YVdf3QT1T4RwdggGbuR8Eakd',
                                                                                                           'Ahgd7ZczsP6XcWuhsF1Hn7i9W31uHMMPkmffrALL7ers',
                                                                                                             'ANUBao2a6xATKoNdFrnDADkjVYpcmLVcPp978RwhkJxf',
                                                                                                               'AobVSwdW9BbpMdJvTqeCN4hPAmh4rHm7vwLnQ5ATSyrS',
                                                                                                                 'ASTyfSima4LLAdDgoFGkgqoKowG1LZFDr9fAQrg7iaJZ',
                                                                                                                   'BJGyq6BDtn4KXrGr1yfBuCXyxToP6CWfA4sYRyDPSEq4', 
                                                                                                                   'BKenVF6yfi2SLpz78JXWm5tv45GFt2VkvnGtQsQoGwwR', 
                                                                                                                   'BunaYnktTigcU1ovzVt9dG7NMv2gW5VX7MBfSS8J38s2', 
                                                                                                                   'BY4StcU9Y2BpgH8quZzorg31EGE4L1rjomN8FNsCBEcx', 
                                                                                                                   'C68a6RCGLiPskbPYtAcsCjhG8tfTWYcoB4JjCrXFdqyo', 
                                                                                                                   'CAo1dCGYrB6NhHh5xb1cGjUiu86iyCfMTENxgHumSve4', 
                                                                                                                   'CDhUgGEiUxx1aTbnoiSAKcmBhnGUFRQ6AMzuLQRD5VFZ', 
                                                                                                                   'CK8i4zFXkDE2KWfyg7g9S748r6mwxajbcKcyGhQMR3qQ', 
                                                                                                                   'CqQ6AX1fiFfHKKY3saGzT5pgbkLwfLVrrAKpFhUG38oe', 
                                                                                                                   'D5LmzoS7PXmT8oMVpSY1iSk1bZD2XLsc9nE5rSmwvyuQ', 'D89hHJT5Aqyx1trP6EnGY9jJUB3whgnq3aUvvCqedvzf', 'D8nrjrTYdU9HjkyvZdheA9sJ5qTK6mf726AGjfKc4bSb', 'DPqsobysNf5iA9w7zrQM8HLzCKZEDMkZsWbiidsAt1xo', 'EqfdXQTL9r8yzjgeTx3nceJBxH3gAMBjiQrUzsf6oRhx', 'ETZY5TjMKdV2KdHVmUNTN56pWhMc8TyjrXtQ7YexDCmG', 'ExB8m6GpZ7hg8NjSDDoEsg6U4XSvWUVD3wLfTLd51xXY', 'FeNayVKekV9FJzhD7ycTd6sbKEyzt9CRCiqxw5nr41yR', 'FfNbZZSMzNvJTwC33NT2fi61vQJ5YUvREfukvVyYTQab', 'FH9iLV5Z8EUEDMnW6CzUPkpDhWJCsHqJ5N4W23njNsUo', 'Fi3kigPLsFtSvq7N1KBtz51C8Gc8XyMFWhYjbRkaFMxS', 'FJxqxWAm6ebfaQ7ewns1SWVaM3Xm5FTkQz1T9aBRGuTn', 'FpwQQhQQoEaVu3WU2qZMfF1hx48YyfwsLoRgXG83E99Q', 'FQz71kJA22skWHLkuWcdDKjRZH3sKGLtSprRMwbMfMW5', 'FWznbcNXWQuHTawe9RxvQ2LdCENssh12dsznf4RiouN5', 'FxteHmLwG9nk1eL4pjNve3Eub2goGkkz6g6TbvdmW46a', 'GCRJD52pGwcCSs4oswYxTBCPatxY1P6WpxCC9R9zty6r', 'GCtXCKbCKWudrAsbrHtd2ngMgrSmGZGjoaYRtzPEF3Lv', 'Gem2VAypSg7Ai7vjDKPTtqFahpoQWkfgVkyzx3rPoTka', 'GJRs4FwHtemZ5ZE9x3FNvJ8TMwitKTh21yxdRPqn7npE', 'GkCyx6dfBqopSqEdQ6mMQH7xJtpdE1yAZEdoGpDPwN7s', 'GLH37YLmPxX5pSkm5E4pKFtP2xfcgANR5TYv892Fj2aE', 'GP1uopJfig6UoPVq4JC3nw9SGsjYbRixNfD5JhiVXuYC', 'H13inzihSqSgfhHrGXNcrhMPzZmp73Mjs1tu8PrnFN7K', 'H8sMJSCQxfKiFTCfDR3DUMLPwcRbM61LGFJ8N4dK3WjS', 'HBxZShcE86UMmF93KUM8eWJKqeEXi5cqWCLYLMMhqMYm', 'HiRpdAZifEsZGdzQ5Xo5wcnaH3D2Jj9SoNsUzcYNK78J', 'HVh6wHNBAsG3pq1Bj5oCzRjoWKVogEDHwUHkRz3ekFgt', 'Hvkm4H2Ta3L3ssWbB5jeC4kpEJDuZnZqapAXp1V7UHEw', 'i57ExrKB2i4mSgjSuq2xz617mQXmu33WG2WEYypmdvX', 'i9XvhQqBCTQapqaFKPDuCbtPYMCwELmX8VTCsDhRG7d', 'iGdFcQoyR2MwbXMHQskhmNsqddZ6rinsipHc4TNSdwu', 'is6MTRHEgyFLNTfYcuV4QBWLjrZBfmhVNYR6ccgr8KV', 'krakeNd6ednDPEXxHAmoBs1qKVM8kLg79PvWF2mhXV1', 'N6Rktaxp2dgizmRfWS3mXbS9uTuKARe6WEA85oSkKYv', 'u6PJ8DtQuPFnfmwHbGFULQ4u4EgjDiyYKjVEsynXq2w', 'WR87TKyUfMDzGojFvRxq2f1xf7Jcgkvoc7Z2tDVLHdY'}

Итерационное сохранение транзакций

In [ ]:
counter=0
proxy_cycle = cycle(proxy_list)
proxy = next(proxy_cycle)
for address in filtered_adresses:
    
    start_human = datetime.strptime('2023-10-01 00:00:00+0000', '%Y-%m-%d %H:%M:%S%z')
    start=to_dt_ms(start_human)
    time_finish=start_human-timedelta(days=1500)
    finish=to_dt_ms(time_finish)
    new_time_start=start
    
    while finish < new_time_start:
        if counter % 1000 == 0:
            proxy = next(proxy_cycle)
        try:
            new_df=save_transfer_proxi(address,0,new_time_start,proxy)
            if "Action" not in new_df.columns:
                proxy = next(proxy_cycle)
                
                print("Неверный CSV — возможно Cloudflare или ошибка Solscan")
                time.sleep(1)
                continue
            if new_df.empty and "Action" in new_df.columns:
                print(f"Нет данных для {address} в этом диапазоне, выходим из цикла.")
                break
            last_human = new_df['Human Time'].iloc[-1]
            try:
                # переводим её в datetime (делаем явный utc на всякий случай)
                last_dt = pd.to_datetime(last_human, utc=True)

                # переводим обратно в ms
                new_time_start = to_dt_ms(last_dt) - 1
            except Exception as e:
                print(f"Ошибка {e}, не смогли перевести время")
                break
            counter+=1
        except Exception as e:
            print(f"Ошибка {e} по прокси {proxy}")
            proxy = next(proxy_cycle)
            continue
            
    time.sleep(5)

Сохранили 1000 ; использовали 103.176.97.12:8082 по 9Z7S8vCj6nDbK9t4m4AU3vZpKm4UufHAwpmRYyKgZf7r с 2023-10-01 00:00:00+00:00 в C:\Users\User\my\py\practise\solana\downloads1\9Z7S8vCj6nDbK9t4m4AU3vZpKm4UufHAwpmRYyKgZf7r\export_transfer_9Z7S8vCj6nDbK9t4m4AU3vZpKm4UufHAwpmRYyKgZf7r_1767133847614_addr-9Z7S8vCj6nDbK9t4m4AU3vZpKm4UufHAwpmRYyKgZf7r_from-0_to-1696118400_amt-1.csv
Нет данных для 9Z7S8vCj6nDbK9t4m4AU3vZpKm4UufHAwpmRYyKgZf7r в этом диапазоне, выходим из цикла.
Сохранили 1000 ; использовали 103.176.97.107:8082 по i9XvhQqBCTQapqaFKPDuCbtPYMCwELmX8VTCsDhRG7d с 2023-10-01 00:00:00+00:00 в C:\Users\User\my\py\practise\solana\downloads1\i9XvhQqBCTQapqaFKPDuCbtPYMCwELmX8VTCsDhRG7d\export_transfer_i9XvhQqBCTQapqaFKPDuCbtPYMCwELmX8VTCsDhRG7d_1767133854125_addr-i9XvhQqBCTQapqaFKPDuCbtPYMCwELmX8VTCsDhRG7d_from-0_to-1696118400_amt-1.csv
Нет данных для i9XvhQqBCTQapqaFKPDuCbtPYMCwELmX8VTCsDhRG7d в этом диапазоне, выходим из цикла.
Ошибка HTTPSConnectionPool(host='api-v2.solscan.io', port=443)